# tabctx quickstart

Fit a table once, keep it as a cached *context* on a server, and predict against it with a scikit-learn-style API. This notebook runs the whole flow end to end: install, start a server (CPU is fine), classify, regress with prediction quantiles, and manage contexts.

## 1. Install

In [1]:
!pip install -q "tabctx[standalone,tabicl,client]"

## 2. Start a server

The server hosts the TabICL model. The first start downloads the TabICLv2 weights (BSD-3, no account needed). We start it in the background and wait until it answers.

In [2]:
import json
import os
import subprocess
import sys
import time
import urllib.request

env = {**os.environ, "TABCTX_BACKEND": "tabicl"}
server = subprocess.Popen(
    [sys.executable, "-m", "tabctx.serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

models = None
for _ in range(180):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as resp:
            models = json.load(resp)
        break
    except Exception:
        time.sleep(5)

assert models is not None, "server did not come up"
models

{'data': [{'id': 'tabicl-v2', 'object': 'model', 'default': True}]}

## 3. Fit once, predict many (classification)

`fit` sends the table to the server, which encodes it into a cached context. Every later `predict` / `predict_proba` reuses that context.

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from tabctx import TabctxClassifier, TabctxClient

X, y = fetch_openml("credit-g", version=1, as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

client = TabctxClient("http://127.0.0.1:8000")
clf = TabctxClassifier(client=client)
clf.fit(X_train, y_train)
print("context id:", clf.dataset_id_)

context id: sklearn-93906881b2b76a20dd1a638b8bf37862


In [4]:
print("accuracy:", clf.score(X_test, y_test))
clf.predict_proba(X_test[:5])

accuracy: 0.772


array([[0.7023325 , 0.29766747],
       [0.05054151, 0.94945848],
       [0.04760469, 0.95239532],
       [0.26294938, 0.73705065],
       [0.39968097, 0.60031903]])

## 4. Regression with prediction quantiles

`TabctxRegressor.predict_quantiles` returns predictive quantiles for each row, not just a point estimate. (The diabetes dataset ships with scikit-learn, so this cell needs no download.)

In [5]:
from sklearn.datasets import load_diabetes

from tabctx import TabctxRegressor

data = load_diabetes(as_frame=True)
Xr, yr = data.data, data.target
Xr_train, Xr_test = Xr.iloc[:-100], Xr.iloc[-100:]
yr_train, yr_test = yr.iloc[:-100], yr.iloc[-100:]

reg = TabctxRegressor(client=client)
reg.fit(Xr_train, yr_train)
print("context id:", reg.dataset_id_)
reg.predict(Xr_test[:5])

context id: sklearn-aea702693a87d2751cc473edb459f246


array([164.15979004, 143.35258484, 152.58087158, 130.20314026,
       195.56369019])

In [6]:
import pandas as pd

quantiles = (0.05, 0.5, 0.95)
q = reg.predict_quantiles(Xr_test[:5], quantiles=quantiles)
pd.DataFrame(q, columns=[f"q{int(q_ * 100)}" for q_ in quantiles])

,q5,q50,q95
0,69.302078,160.332001,269.507080
1,55.275356,138.298859,248.152664
2,62.262737,149.525345,253.321518
3,52.836693,122.921402,230.515656
4,94.553528,195.324875,295.893463


## 5. List and delete contexts

Contexts live on the server until they are evicted or deleted. `client.datasets()` lists them; `client.delete(...)` removes one.

In [7]:
[d["dataset_id"] for d in client.datasets()]

['sklearn-93906881b2b76a20dd1a638b8bf37862',
 'sklearn-aea702693a87d2751cc473edb459f246']

In [8]:
client.delete(clf.dataset_id_)
client.delete(reg.dataset_id_)
client.datasets()

[]

## 6. Shut the server down

In [9]:
server.terminate()